In [ ]:


# Appending the extra C_switches calculated to shift the center frequency
C_sw_min_freq = 1612.48e-15 # From earlier: Switch capacitance for 6.0 GHz fmin point
C_sw_max_freq = 200.30e-15  # From earlier: Switch capacitance for 9.5 GHz fmax point


# --- 1. UNIT PARSING & LOADING ---
unit_map = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}

def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() == "error" or str(value).strip() == "": return np.nan
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', str(value).strip())
    if match: return float(match.group(1)) * unit_map.get(match.group(2), 1.0)
    return np.nan

# --- File Paths (Only Compensated Configurations) ---
fmin_files_comp = {
    'ind': "ind_at_6Ghz_MEMOMEMO_PrimeSim_default_Measurements_history_1_20260602_17_18_30.19.csv",
    'cc': "CC_max_PrimeSim_default_justCC_Measurements_history_1_20260410_16_41_15.9.csv",
    'var': "TOTAL_VTEMP_REAL_CORNERS_LC_NEWVAR_F_PrimeSim_default_VtempReal_Measurements_history_1_20260615_19_17_31.60.csv"
}

fmax_files_comp = {
    'ind': "ind_at_9.5Ghz_MEMO.csv",
    'cc': "CC_min_PrimeSim_default_justCC_Measurements_history_1_20260410_16_56_40.57.csv",
    'var': "TOTAL_VTEMP_REAL_CORNERS_LC_NEWVAR_F_PrimeSim_default_VtempReal_Measurements_history_1_20260615_19_17_31.60.csv"
}

# --- Configuration Mapping ---
combinations_comp = {
    'Wcq (Worst)': ('wcQ', 'wcQ125', 'SS'),
    'Bcq (Best)': ('bcQ', 'bcQ-40', 'FF'),
    'Typical': ('TT', 'Typ25', 'TT')
}

colors = {'Wcq (Worst)': 'red', 'Bcq (Best)': 'blue', 'Typical': 'green'}
C_parasitic = 114.5e-15 # 114.5 fF

# Global dictionaries to store plot traces
plot_lines_f = {}
plot_lines_q = {}
target_temps = [-40, 25, 125]

# --- Dynamic Data Processing Function ---
def process_data(files, context_tag=""):
    if not all(os.path.exists(f) for f in files.values()):
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # 1. Parse Inductor Data
    df_i_raw = pd.read_csv(files['ind'])
    l_col = next((c for c in df_i_raw.columns if any(p in c for p in ['L_min', 'L_max', 'L_measured', 'L:'])), df_i_raw.columns[-2])
    q_ind_col = next((c for c in df_i_raw.columns if any(p in c for p in ['Q:', 'Q_factor', 'Q_measured', 'Q_ind'])), df_i_raw.columns[-1])
    t_ind_col = next((c for c in df_i_raw.columns if 'temp' in c.lower()), df_i_raw.columns[1])
    
    ind_data = []
    for _, row in df_i_raw.iterrows():
        c_str = str(row['Corner']).lower()
        key = 'TT' if 'tt' in c_str else 'bcQ' if 'bcq' in c_str else 'wcQ' if 'wcq' in c_str else None
        if key:
            ind_data.append({
                'Corner': key, 
                'Temp': round(float(parse_units(row[t_ind_col])), 1), 
                'L': parse_units(row[l_col]), 
                'Q_ind': float(row[q_ind_col]) if q_ind_col else 100.0
            })
    df_ind = pd.DataFrame(ind_data)

    # 2. Parse Cap Bank Data
    df_c_raw = pd.read_csv(files['cc'])
    cc_col = next((c for c in df_c_raw.columns if any(p in c for p in ['Cap_measured', 'Cap_min', 'Cap_max', 'CC', 'Cap_'])), df_c_raw.columns[-1])
    t_cc_col = next((c for c in df_c_raw.columns if 'temp' in c.lower()), df_c_raw.columns[1])
    
    df_cc = pd.DataFrame({
        'Corner': df_c_raw['Corner'].astype(str),
        'Temp': df_c_raw[t_cc_col].apply(lambda x: round(float(parse_units(x)), 1)),
        'CC': df_c_raw[cc_col].apply(parse_units)
    })

    # 3. Parse Varactor Data
    df_v_raw = pd.read_csv(files['var'])
    t_var_col = next((c for c in df_v_raw.columns if 'temp' in c.lower() or 'sweep' in c.lower()), df_v_raw.columns[1])
    cvar_col = next((c for c in df_v_raw.columns if any(p in c for p in ['Cap_max', 'Cap_measured', 'Cvar', 'Cap_', 'Cap_measuredAt1Mhz'])), df_v_raw.columns[2])
    q_var_col = next((c for c in df_v_raw.columns if any(p in c for p in ['Q_factor', 'Q_factorAt1Mhz', 'Q:', 'Q_measured'])), df_v_raw.columns[3])

    df_var = pd.DataFrame({
        'Corner': df_v_raw['Corner'].astype(str),
        'Temp': df_v_raw[t_var_col].apply(lambda x: round(float(parse_units(x)), 1) if isinstance(x, str) else round(float(x), 1)),
        'Cvar': df_v_raw[cvar_col].apply(parse_units),
        'Q_var': df_v_raw[q_var_col].apply(parse_units) if q_var_col else 100.0
    }).dropna()

    return df_ind, df_cc, df_var

# --- Subset Filtering Selector ---
def get_clean_subset(df, key):
    k_lower = key.lower()
    sub = df[df['Corner'].str.lower() == k_lower]
    if not sub.empty: return sub
    
    sub = df[df['Corner'].str.lower().str.contains(k_lower) | df['Corner'].str.lower().apply(lambda x: k_lower in x)]
    if not sub.empty: return sub
    
    base_key = re.sub(r'\d+$', '', re.sub(r'-\d+$', '', k_lower))
    sub = df[df['Corner'].str.lower().str.contains(base_key) | df['Corner'].str.lower().apply(lambda x: base_key in x)]
    if not sub.empty: return sub

    sub = df[df['Corner'].str.lower().apply(lambda x: x in k_lower)]
    return sub

# --- Analyzer Function ---
def analyze_and_store(files, combinations_map, label_suffix, linestyle, is_compensated, summary_list, C_switches):
    df_ind, df_cc, df_var = process_data(files, context_tag=label_suffix)
    if df_ind.empty or df_cc.empty or df_var.empty: 
        return

    for label, (l_key, cc_key, v_key) in combinations_map.items():
        i_sub = get_clean_subset(df_ind, l_key)
        cc_sub = get_clean_subset(df_cc, cc_key)
        v_sub = get_clean_subset(df_var, v_key)
        
        if i_sub.empty or cc_sub.empty or v_sub.empty: 
            continue

        i_sub = i_sub.groupby('Temp', as_index=False).mean(numeric_only=True).sort_values('Temp')
        cc_sub = cc_sub.groupby('Temp', as_index=False).mean(numeric_only=True).sort_values('Temp')
        v_sub = v_sub.groupby('Temp', as_index=False).mean(numeric_only=True).sort_values('Temp')

        if len(i_sub) < 2 or len(cc_sub) < 2 or len(v_sub) < 2:
            continue

        f_L = interp1d(i_sub['Temp'], i_sub['L'], fill_value="extrapolate")
        f_Qi = interp1d(i_sub['Temp'], i_sub['Q_ind'], fill_value="extrapolate")
        f_CC = interp1d(cc_sub['Temp'], cc_sub['CC'], fill_value="extrapolate")
        f_Cvar = interp1d(v_sub['Temp'], v_sub['Cvar'], fill_value="extrapolate")
        f_Qv = interp1d(v_sub['Temp'], v_sub['Q_var'], fill_value="extrapolate")

        t_range = np.linspace(-40, 125, 100)
        L_v, Qi_v, CC_v, Cv_v, Qv_v = f_L(t_range), f_Qi(t_range), f_CC(t_range), f_Cvar(t_range), f_Qv(t_range)
        
        C_tot_v = CC_v + Cv_v + C_parasitic + C_switches
        
        L_v = np.where(L_v <= 0, 1e-22, L_v)
        C_tot_v = np.where(C_tot_v <= 0, 1e-22, C_tot_v)
        Qi_v = np.where(Qi_v <= 0, 1e-22, Qi_v)
        Qv_v = np.where(Qv_v <= 0, 1e-22, Qv_v)

        freq = (1 / (2 * np.pi * np.sqrt(L_v * C_tot_v))) / 1e9
        q_tot = (Qi_v * Qv_v) / (Qi_v + Qv_v)
        
        full_label = f"{label} ({label_suffix})"
        plot_lines_f[full_label] = (t_range, freq, colors[label], linestyle)
        plot_lines_q[full_label] = (t_range, q_tot, colors[label], linestyle)

        res = {'Label': full_label}
        for t in target_temps:
            Ct = max(f_CC(t) + f_Cvar(t) + C_parasitic + C_switches, 1e-22)
            Lt = max(f_L(t), 1e-22)
            res[f'f_{t}'] = (1 / (2 * np.pi * np.sqrt(Lt * Ct))) / 1e9
            res[f'q_{t}'] = (f_Qi(t) * f_Qv(t)) / (f_Qi(t) + f_Qv(t))
        
        f_sw = freq
        delta_freq_ghz_mag = np.max(f_sw) - np.min(f_sw)
        
        dir_C = -1 if C_tot_v[-1] < C_tot_v[0] else 1
        
        res['f_delta_ghz'] = delta_freq_ghz_mag
        res['f_delta_mhz'] = delta_freq_ghz_mag * 1000
        res['f_var_pct'] = (delta_freq_ghz_mag / np.mean(f_sw)) * 100
        res['L_var_pct'] = ((np.max(L_v) - np.min(L_v)) / np.mean(L_v)) * 100
        res['C_var_pct'] = (((np.max(C_tot_v) - np.min(C_tot_v)) / np.mean(C_tot_v)) * 100) * dir_C
            
        summary_list.append(res)

summary_comp_fmin = []
summary_comp_fmax = []

# --- Extract Data Runs (Compensated Only) ---
analyze_and_store(fmin_files_comp, combinations_comp, "Compensated fmin", "-.", True, summary_comp_fmin, C_sw_min_freq)
analyze_and_store(fmax_files_comp, combinations_comp, "Compensated fmax", "-", True, summary_comp_fmax, C_sw_max_freq)


# ==========================================
# --- PLOTTING (Separated Windows) ---
# ==========================================

# Window 1: Frequency Plot
fig_f, ax_f = plt.subplots(figsize=(10, 7))

for label, (x, y, color, style) in plot_lines_f.items():
    ax_f.plot(x, y, label=label, color=color, linestyle=style, linewidth=2)

ax_f.set_title("Compensated Topology: Tank Resonance Frequency Overlap (Fmin & Fmax)", fontweight='bold')
ax_f.set_ylabel("Frequency (GHz)")
ax_f.set_xlabel("Temperature (°C)")
ax_f.grid(True, linestyle='--', alpha=0.6)
ax_f.legend(loc='best')

fig_f.tight_layout()

# Window 2: Quality Factor Plot
fig_q, ax_q = plt.subplots(figsize=(10, 7))

for label, (x, y, color, style) in plot_lines_q.items():
    ax_q.plot(x, y, label=label, color=color, linestyle=style, linewidth=2)

ax_q.set_title(r"Compensated Topology: Tank Total Quality Factor ($Q_{total} = \frac{Q_L Q_{Cvar}}{Q_L + Q_{Cvar}}$)", fontweight='bold')
ax_q.set_ylabel("Total Q")
ax_q.set_xlabel("Temperature (°C)")
ax_q.grid(True, linestyle='--', alpha=0.6)
ax_q.legend(loc='best')

fig_q.tight_layout()

# Display both windows
plt.show()


# --- PRINTING SUMMARY TABLES ---
def print_table(summary_points, title):
    print("\n" + "="*145)
    print(f" {title}")
    print("="*145)
    if not summary_points:
        print(" [NO DATA EXTRACTED - Check column titles or internal row properties within the source csv files]")
        print("="*145)
        return
    print(f"{'Corner Combination':<30} | {'Temp':<6} | {'Freq (GHz)':<12} | {'Total Q':<10} | {'Delta Freq':<12} | {'%Dev Freq':<10} | {'L Var %':<10} | {'C Var %':<10}")
    print("-" * 145)
    for res in summary_points:
        for i, t in enumerate(target_temps):
            l_str = res['Label'] if i == 1 else ""
            d_str = f"{res['f_delta_mhz']:>7.2f} MHz" if i == 1 else ""
            freq_val = res[f'f_{t}']
            freq_dev_pct = (res['f_delta_ghz'] / freq_val) * 100 if freq_val else np.nan
            
            l_var_str = f"{res['L_var_pct']:>8.3f}%" if i == 1 else ""
            c_var_str = f"{res['C_var_pct']:>8.3f}%" if i == 1 else ""

            print(f"{l_str:<30} | {t:>4}°C | {freq_val:>10.4f}   | {res[f'q_{t}']:>8.2f}   | {d_str:<12} | {freq_dev_pct:>9.3f}% | {l_var_str:<10} | {c_var_str:<10}")
        print(f"{'':<30} | %Var | {res['f_var_pct']:>9.3f}%   | {'':<10} | {'':<12} | {'':<10} | {'':<10} | {'':<10}")
        print("-" * 145)

print_table(summary_comp_fmin, "COMPENSATED TOPOLOGY SUMMARY (FMIN)")
print_table(summary_comp_fmax, "COMPENSATED TOPOLOGY SUMMARY (FMAX)")